### PERFORM STRUCTUCTED STRAMING FOR `CUSTOMERS`

#### DEFINE SCHEMA FOR `CUSTOMERS.JSON`

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DateType


customers_schema = StructType(
    fields=[StructField('customer_id', IntegerType()),
            StructField('customer_name', StringType()),
            StructField('date_of_birth', DateType()),
            StructField('telephone', StringType()),
            StructField('email', StringType()),
            StructField('member_since', DateType()),
            StructField('created_timestamp', TimestampType())
            ]
)

In [0]:
customers_streaming_df = (spark.readStream
                              .format('cloudFiles')
                              .option('cloudFiles.format', 'json')
                            # .option('cloudFiles.schemaLocation', '/Volumes/gizmo/landing/operational_data/customers_streaming/customers_schema/')
                              .option('cloudFiles.inferColumnTypes', 'true')
                            #   .option('cloudFiles.schemaHints', 'customer_id:long')
                              # .option('pathGlobFilter', 'customers_2024_*.*')  # INCLUDED FILES TO BE PROCESSED
                              .option('pathGlobFilter', 'customers_{2024,2025}_*.*')  # INCLUDED FILES TO BE PROCESSED
                              .option('cloudFiles.schemaEvolutionMode', 'rescue')
                              .schema(customers_schema)
                              .load('/Volumes/gizmo/landing/operational_data/customers_streaming/files/')
                          )

In [0]:
from pyspark.sql.functions import current_timestamp, col

customers_trans_streaming_df = (customers_streaming_df
            .withColumn('load_timestamp', current_timestamp())
            .withColumn('file_name', col("_metadata.file_path")) 
)

# display(customers_trans_streaming_df)

In [0]:
customers_streaming_qry_df = (
    customers_trans_streaming_df
    .writeStream
    .format('delta')
    .queryName('CUSTOMERS_STREAMING_BRONZE_INGESTION_QUERY')
    .trigger(availableNow=True)
    .outputMode('append')
    .option('checkpointLocation', '/Volumes/gizmo/landing/operational_data/customers_streaming/customers_checkpoint/')
    .option('mergeSchema', 'true')
    .toTable('GIZMO.BRONZE.CUSTOMERS_STREAMING')
)

In [0]:
dbutils.notebook.exit('SUCCESSFULLY LOADED INTO GIZMO.BRONZE.CUSTOMERS_STREAMING')